<a href="https://colab.research.google.com/github/Skquark/MissingLink/blob/main/notebooks/Pixal3D_Wheel_Builder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Pixal3D — CUDA Wheel Builder

Run this notebook once per GPU type to precompile the four CUDA extension packages that Pixal3D depends on. The resulting `.whl` files get saved to Google Drive and can be uploaded to GitHub Releases for use in the main notebook.

### Packages Built
| Package | Source | Purpose |
|---------|--------|---------|
| `o_voxel` | microsoft/TRELLIS.2 | Mesh ↔ voxel conversion |
| `cumesh` | JeffreyXiang/CuMesh | CUDA mesh post-processing |
| `flex_gemm` | JeffreyXiang/FlexGEMM | Sparse conv via Triton |
| `nvdiffrec_render` | JeffreyXiang/nvdiffrec | PBR renderer (optional) |

### GPU Architectures
| GPU | `TORCH_CUDA_ARCH_LIST` |
|-----|------------------------|
| T4 | `7.5` |
| A100 | `8.0` |
| L4 | `8.9` |

### Steps
1. Connect to the desired GPU type
2. Mount Google Drive (Step 2)
3. Run all cells in order
4. Download wheels from `/content/drive/MyDrive/pixal3d_wheels/`
5. Upload to GitHub Releases → create a tag like `wheels-v1`

In [ ]:
# @title Step 1 — Auto-Detect GPU & Set Architecture

import os, sys
import torch

gpu_name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
arch = f'{cap[0]}.{cap[1]}'
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3

os.environ['TORCH_CUDA_ARCH_LIST'] = arch
os.environ['CUDA_HOME'] = os.environ.get('CUDA_HOME', '/usr/local/cuda')

print(f'[GPU] {gpu_name} ({vram:.0f} GB)')
print(f'[CUDA] Arch: sm_{arch.replace(".", "")} — TORCH_CUDA_ARCH_LIST={arch}')
print(f'[CUDA] CUDA_HOME={os.environ["CUDA_HOME"]}')
print(f'[Torch] {torch.__version__} + CUDA {torch.version.cuda}')

In [ ]:
# @title Step 2 — Mount Google Drive & Set Up Output Directory

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

gpu_tag = {'7.5': 't4', '8.0': 'a100', '8.9': 'l4'}.get(arch, 'unknown')
WHEEL_DIR = f'/content/drive/MyDrive/pixal3d_wheels/{gpu_tag}'
os.makedirs(WHEEL_DIR, exist_ok=True)
print(f'[Output] Wheels will be saved to {WHEEL_DIR}/')
print(f'[Output] GPU tag: {gpu_tag}')

In [ ]:
# @title Step 3 — Install Build Dependencies

!pip install -q ninja setuptools wheel build
print('[Ninja] Installed')
import sys
# Show the exact torch version + path (important for ABI matching)
cxx_abi = 1 if torch.compiled_with_cxx11_abi() else 0
print(f'[Torch ABI] _GLIBCXX_USE_CXX11_ABI={cxx_abi}')

In [ ]:
# @title Step 4 — Clone TRELLIS.2 (needed for o_voxel source)
# @markdown We only need the `o-voxel/` subdirectory and its third-party deps.

import os, shutil

os.chdir('/content')
if os.path.exists('/content/trellis2_src'):
    shutil.rmtree('/content/trellis2_src')

!git clone --depth 1 https://github.com/microsoft/TRELLIS.2.git /content/trellis2_src
os.chdir('/content/trellis2_src/o-voxel')
# Clone eigen from GitLab (needed by o-voxel CUDA headers)
if not os.path.exists('third_party/eigen/Eigen'):
    !git clone --depth 1 https://gitlab.com/libeigen/eigen.git third_party/eigen
print('[TRELLIS.2] Cloned — o-voxel source ready')
print(f'  Path: /content/trellis2_src/o-voxel/')

In [ ]:
# @title Step 5 — Build o_voxel Wheel
# @markdown Compiles the mesh↔O-Voxel conversion library. ~4 min on A100.

import subprocess, glob, shutil

SRC = '/content/trellis2_src/o-voxel'

!echo "=== o_voxel build ==="
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'wheel', '--no-build-isolation',
     '--wheel-dir', WHEEL_DIR, SRC],
    cwd=SRC, capture_output=False, text=True
)

if result.returncode == 0:
    whls = glob.glob(os.path.join(WHEEL_DIR, 'o_voxel*.whl'))
    print(f'\n[o_voxel] OK — {len(whls)} wheel(s) in {WHEEL_DIR}/')
    for w in whls:
        print(f'  {os.path.basename(w)} ({os.path.getsize(w)/1024:.0f} KB)')
else:
    print(f'\n[o_voxel] FAILED with return code {result.returncode}')

In [ ]:
# @title Step 6 — Build cumesh Wheel
# @markdown Compiles CUDA mesh utilities (simplify, remesh, atlas). ~5 min on A100.

import subprocess, glob, shutil

SRC = '/content/cumesh_src'
if os.path.exists(SRC):
    shutil.rmtree(SRC)

!git clone --recursive https://github.com/JeffreyXiang/CuMesh.git {SRC}

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'wheel', '--no-build-isolation',
     '--wheel-dir', WHEEL_DIR, SRC],
    cwd=SRC, capture_output=False, text=True
)

if result.returncode == 0:
    whls = glob.glob(os.path.join(WHEEL_DIR, 'cumesh*.whl'))
    print(f'\n[cumesh] OK — {len(whls)} wheel(s) in {WHEEL_DIR}/')
    for w in whls:
        print(f'  {os.path.basename(w)} ({os.path.getsize(w)/1024:.0f} KB)')
else:
    print(f'\n[cumesh] FAILED with return code {result.returncode}')

In [ ]:
# @title Step 7 — Build flex_gemm Wheel
# @markdown Compiles sparse convolution kernels (Triton + CUDA). ~6 min on A100.

import subprocess, glob, shutil

SRC = '/content/flexgemm_src'
if os.path.exists(SRC):
    shutil.rmtree(SRC)

!git clone --recursive https://github.com/JeffreyXiang/FlexGEMM.git {SRC}

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'wheel', '--no-build-isolation',
     '--wheel-dir', WHEEL_DIR, SRC],
    cwd=SRC, capture_output=False, text=True
)

if result.returncode == 0:
    whls = glob.glob(os.path.join(WHEEL_DIR, 'flex_gemm*.whl'))
    print(f'\n[flex_gemm] OK — {len(whls)} wheel(s) in {WHEEL_DIR}/')
    for w in whls:
        print(f'  {os.path.basename(w)} ({os.path.getsize(w)/1024:.0f} KB)')
else:
    print(f'\n[flex_gemm] FAILED with return code {result.returncode}')

In [ ]:
# @title Step 8 — Build nvdiffrec_render Wheel (Optional)
# @markdown PBR split-sum renderer. Often fails on CUDA 12.8 — skip if it does.

import subprocess, glob, shutil

SRC = '/content/nvdiffrec_src'
if os.path.exists(SRC):
    shutil.rmtree(SRC)

!git clone -b renderutils --single-branch https://github.com/JeffreyXiang/nvdiffrec.git {SRC}

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'wheel', '--no-build-isolation',
     '--wheel-dir', WHEEL_DIR, SRC],
    cwd=SRC, capture_output=False, text=True
)

if result.returncode == 0:
    whls = glob.glob(os.path.join(WHEEL_DIR, 'nvdiffrec_render*.whl'))
    print(f'\n[nvdiffrec_render] OK — {len(whls)} wheel(s) in {WHEEL_DIR}/')
    for w in whls:
        print(f'  {os.path.basename(w)} ({os.path.getsize(w)/1024:.0f} KB)')
else:
    print(f'\n[nvdiffrec_render] FAILED (expected on CUDA 12.8 — use nvdiffrast instead)')

In [ ]:
# @title Step 9 — Summary & Verify

import os, glob

py_ver = f'cp{sys.version_info.major}{sys.version_info.minor}'

print(f'GPU: {gpu_name}')
print(f'Tag: {gpu_tag}  |  Arch: sm_{arch.replace(".", "")}')
print(f'PyTorch: {torch.__version__} (CUDA {torch.version.cuda})')
print(f'CXX11 ABI: {1 if torch.compiled_with_cxx11_abi() else 0}')
print()

all_whls = sorted(glob.glob(os.path.join(WHEEL_DIR, '*.whl')))
if all_whls:
    print(f'=== Built {len(all_whls)} wheel(s) in {WHEEL_DIR}/ ===')
    total = 0
    for w in all_whls:
        sz = os.path.getsize(w)
        total += sz
        print(f'  {os.path.basename(w):50s} {sz/1024:8.0f} KB')
    print(f'  {"TOTAL":50s} {total/1024:8.0f} KB')
    print()
    print('### Upload to GitHub Releases ###')
    print(f'1. Create a release at: https://github.com/Skquark/MissingLink/releases/new')
    print(f'2. Tag: wheels/{gpu_tag}-v1')
    print(f'3. Upload all {len(all_whls)} .whl files from {WHEEL_DIR}/')
    print(f'4. Update the Pixal3D notebook Step 1 to point at the release URLs')
else:
    print('[FAIL] No wheels were built. Check errors in Steps 5-8 above.')

### Integration into Pixal3D Notebook (Step 1)

Once wheels are uploaded to GitHub Releases, you can replace the MissingLink token dependency. Here's the working replacement for Step 1 in the main notebook:

```python
# ====================================================================
# STEP 1 — Install Dependencies (Free Wheels)
# ====================================================================
# GPU-specific prebuilt wheels hosted on GitHub Releases.
# ====================================================================
import os, sys, torch

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

# Auto-detect GPU
gpu_name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
arch = f'{cap[0]}.{cap[1]}'
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
gpu_tag = {'7.5': 't4', '8.0': 'a100', '8.9': 'l4'}.get(arch, 't4')
print(f'[GPU] Detected {gpu_name} ({vram_gb:.0f} GB)')

# Install prebuilt wheels from GitHub Releases
# Replace WHEELS_TAG with the actual release tag (e.g. wheels/a100-v1)
WHEELS_TAG = f'wheels/{gpu_tag}-v1'
RELEASE = f'https://github.com/Skquark/MissingLink/releases/download/{WHEELS_TAG}'

import subprocess as _sp
# Install core CUDA packages (o_voxel, cumesh, flex_gemm)
for name in ['o_voxel', 'cumesh', 'flex_gemm']:
    url = f'{RELEASE}/{name}-0.0.0-py3-none-any.whl'
    print(f'  Installing {name}...')
    _sp.run(['pip', 'install', '-q', '--no-deps', url], check=True)

print('[CUDA Packages] Installed')

# nvdiffrast (compile from source — same as before)
!pip install -q ninja
os.environ['TORCH_CUDA_ARCH_LIST'] = arch
_sp.run(['pip', 'install', '--no-build-isolation', 'git+https://github.com/NVlabs/nvdiffrast.git'], check=True)
print('[nvdiffrast] Done')

# Python dependencies
!pip install -q trimesh pygltflib plyfile moderngl huggingface_hub kornia kornia-rs \
    imageio imageio-ffmpeg tqdm easydict opencv-python-headless \
    zstandard timm diffusers accelerate gradio einops onnxruntime
!pip install -q git+https://github.com/microsoft/MoGe.git

# natten (prebuilt wheel)
try:
    !pip install natten==0.21.6+torch2110cu128 -f https://whl.natten.org -q
    import natten
    natten_ok = getattr(natten, 'HAS_LIBNATTEN', False)
except Exception:
    natten_ok = False
    print('  natten - skipped (NAF disabled)')

!pip install -q --force-reinstall --no-deps \
    https://github.com/LDYang694/Storages/releases/download/20260430/utils3d-0.0.2-py3-none-any.whl

os.makedirs('/content/images_in', exist_ok=True)
print(f'\n[DONE] {gpu_name} ({vram_gb:.0f} GB) — all dependencies installed')
```

### Naming Convention

The GitHub Release should use tags like `wheels/a100-v1`, `wheels/t4-v1`, `wheels/l4-v1`. The wheel filenames may differ slightly from `0.0.0-py3-none-any` — check the actual filenames in Drive and update the URLs above. You can also upload a `requirements.txt` to the release and use:

```python
!pip install -q --no-deps -r {RELEASE}/requirements.txt
```

### Keeping MissingLink as Fallback

To keep MissingLink as an option: set a `TOKEN` Colab form field. If provided, use MissingLink; if empty, fall back to GitHub Releases.